# Notebook 03

In [1]:
# in this notebook I'm trying to make datasets smaller 

In [2]:
from pathlib import Path
import json
import re
import hashlib
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
from tqdm.auto import tqdm

# I use the same project directory as Notebooks 01 and 02.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")

# I locate the spatial datasets created in Notebook 02.
SPATIAL = ROOT / "04_Spatial_Analysis"

# I organize my optimized datasets and validation files in separate folders.
OUT = ROOT / "05_Optimized_Data"
GEO = OUT / "GeoParquet"
CSV = OUT / "CSV"
AUDIT = OUT / "Audit"

# I create the output folders if they do not already exist.
for folder in (GEO, CSV, AUDIT):
    folder.mkdir(parents=True, exist_ok=True)

# I disable redundant classification exports because the master datasets
# already contain the well and pipeline classifications needed for analysis.
EXPORT_WELL_SYMNUM_LAYERS = False
EXPORT_PIPELINE_STATUS_LAYERS = False
EXPORT_PIPELINE_SYSTEM_LAYERS = False

# I retain point CSV exports for mapping and compatibility with other tools.
EXPORT_POINT_CSV = True

# I retain the expected record counts from Notebook 02 to identify
# accidental record loss during data cleaning and optimization.
EXPECTED = {
    "wind": 19943,
    "battery": 200,
    "wells": 1380352,
    "solar_project_parts": 197,
    "solar_projects": 187,
    "pipeline_source_records": 496305
}

# I retain the reference mileage and solar acreage from Notebook 02
# to verify that the optimization process preserves these measurements.
EXPECTED_PIPELINE_COUNTY_MILES = 362845.156062
EXPECTED_SOLAR_PROJECT_ACRES = 143025.17

# I identify the spatial datasets and audit summaries created in Notebook 02.
SOURCES = {
    "wind": SPATIAL / "Texas_Wind_By_County.gpkg",
    "battery": SPATIAL / "Texas_Battery_By_County.gpkg",
    "wells": SPATIAL / "Texas_Wells_By_County.gpkg",
    "solar_parts": SPATIAL / "Texas_Solar_By_County.gpkg",
    "solar_unique": SPATIAL / "Texas_Solar_Unique_Footprint_By_County.gpkg",
    "pipeline_parts": SPATIAL / "Texas_Pipelines_By_County.gpkg",
    "solar_summary": SPATIAL / "Texas_Solar_County_Summary.csv",
    "pipeline_status_summary": SPATIAL / "Texas_Pipelines_County_Status_Summary.csv",
    "well_category_audit": SPATIAL / "Texas_Wells_Category_County_Audit.csv"
}

# I distinguish the six required spatial datasets from optional audit files.
required_sources = {
    "wind",
    "battery",
    "wells",
    "solar_parts",
    "solar_unique",
    "pipeline_parts"
}

# I check that all required Notebook 02 outputs exist before processing.
# I stop if any are missing because continuing would make the analysis incomplete.
missing = [
    str(path)
    for key, path in SOURCES.items()
    if key in required_sources and not path.exists()
]

if missing:
    raise FileNotFoundError(
        "I need the following Notebook 02 outputs before continuing:\n"
        + "\n".join(missing)
    )

print("I located all six required spatial files for the five included infrastructure types.")
print("Output directory:", OUT)
print("Optional well classification exports:", EXPORT_WELL_SYMNUM_LAYERS)
print("Optional pipeline status exports:", EXPORT_PIPELINE_STATUS_LAYERS)
print("Optional pipeline system exports:", EXPORT_PIPELINE_SYSTEM_LAYERS)

I located all six required spatial files for the five included infrastructure types.
Output directory: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\05_Optimized_Data
Optional well classification exports: False
Optional pipeline status exports: False
Optional pipeline system exports: False


In [3]:
# I initialize the validation and export logs for this notebook run.
CHECKS = []
EXPORT_LOG = []


def check(condition, label, detail=None):
    # I stop the analysis when a validation fails rather than silently continuing.
    passed = bool(condition)
    CHECKS.append({
        "check": label,
        "passed": passed,
        "detail": str(detail) if detail is not None else ""
    })
    if not passed:
        raise ValueError(f"Validation failed: {label}. Details: {detail}")


def required(frame, columns, label):
    # I verify that all essential columns are available before processing.
    missing_columns = [c for c in columns if c not in frame.columns]
    check(not missing_columns, f"{label}: required columns", missing_columns)


def identifier_audit(frame, column, label, unique=True):
    # I check missing identifiers and only require uniqueness when appropriate.
    required(frame, [column], label)
    missing_ids = frame[column].isna().sum()
    check(missing_ids == 0, f"{label}: identifiers present", missing_ids)

    if unique:
        duplicates = frame[column].duplicated().sum()
        check(duplicates == 0, f"{label}: unique identifiers", duplicates)


def pick(frame, columns, rename=None):
    # I retain the selected fields and geometry without modifying the source.
    selected = [c for c in columns if c in frame.columns]
    missing_columns = [c for c in columns if c not in frame.columns]

    if missing_columns:
        print("Optional columns not available:", missing_columns)

    if isinstance(frame, gpd.GeoDataFrame):
        geometry_column = frame.geometry.name
        if geometry_column not in selected:
            selected.append(geometry_column)

    result = frame[selected].copy()

    if rename:
        result = result.rename(columns=rename)

    return result


def same_distribution(original, optimized, columns, label):
    # I compare category counts to detect records lost during optimization.
    for column in columns:
        before = original[column].value_counts(dropna=False)
        after = optimized[column].value_counts(dropna=False)

        check(
            before.equals(after),
            f"{label}: preserved distribution of {column}"
        )


def save_geo(name, frame, category, allowed_geometry):
    # I separate map-ready geometries from missing or unsupported geometries.
    valid_mask = (
        frame.geometry.notna()
        & ~frame.geometry.is_empty
        & frame.geom_type.isin(allowed_geometry)
    )

    ready = frame.loc[valid_mask].copy()
    excluded_count = int((~valid_mask).sum())

    # I preserve the coordinate reference system and save the optimized layer.
    path = GEO / f"{name}.parquet"
    ready.to_parquet(path, index=False)

    # I record how many source rows were mapped or excluded.
    EXPORT_LOG.append({
        "name": name,
        "category": category,
        "path": str(path),
        "source_rows": len(frame),
        "mapped_rows": len(ready),
        "non_map_ready_rows": excluded_count
    })

    print(f"Saved {name}: {len(ready):,} mapped records")
    return ready


def save_csv(name, frame, point_coordinates=False):
    # I export tabular attributes without unnecessarily duplicating geometry.
    result = frame.copy()

    if isinstance(result, gpd.GeoDataFrame):
        if point_coordinates:
            # I include coordinates only for point datasets.
            check(
                result.geom_type.eq("Point").all(),
                f"{name}: point geometries required for coordinates"
            )
            geographic = result.to_crs("EPSG:4326")
            result["longitude"] = geographic.geometry.x
            result["latitude"] = geographic.geometry.y

        result = pd.DataFrame(result.drop(columns=frame.geometry.name))

    path = CSV / f"{name}.csv"
    result.to_csv(path, index=False)

    print(f"Saved {name}: {len(result):,} rows")
    return path


print("Notebook 03 validation and export functions are ready.")

Notebook 03 validation and export functions are ready.


## 2. Wind turbines

In [4]:
wind = gpd.read_file(SOURCES["wind"])
required(wind,["case_id","county_geoid"],"wind")
identifier_audit(wind,"case_id","wind")
check(len(wind)==EXPECTED["wind"],"wind: Notebook 02 count",len(wind))
fields = ["case_id","p_name","p_year","t_manu","t_model","t_cap","t_hh","t_rd",
          "county_geoid","county_name","county_match_status","source_row_id"]
wind_small = pick(wind,fields,{"case_id":"turbine_id","p_name":"project_name",
                               "p_year":"installation_year","t_cap":"capacity_kw"})
check(wind_small["turbine_id"].equals(wind["case_id"]),"wind: turbine identifiers preserved")
same_distribution(wind,wind_small,["county_geoid"],"wind")
wind_ready = save_geo("Texas_Wind_Turbines",wind_small,"wind",["Point"])
check(len(wind_ready)==len(wind),"wind: no geometry exclusions")
if EXPORT_POINT_CSV: save_csv("Texas_Wind_Turbines_Points",wind_ready,True)
print("Wind turbines:",len(wind_ready))
del wind,wind_small,wind_ready

Saved Texas_Wind_Turbines: 19,943 mapped records
Saved Texas_Wind_Turbines_Points: 19,943 rows
Wind turbines: 19943


## 3. Battery energy storage

In [5]:
battery = gpd.read_file(SOURCES["battery"])
required(battery,["Plant Code","county_geoid"],"battery")
identifier_audit(battery,"Plant Code","battery")
check(len(battery)==EXPECTED["battery"],"battery: Notebook 02 count",len(battery))
fields = ["Plant Code","Plant Name","plant_name","battery_generators","capacity_mw",
          "County","county","county_geoid","county_name","county_match_status","source_row_id"]
battery_small = pick(battery,fields,{"Plant Code":"plant_id","battery_generators":"generator_count",
                                     "capacity_mw":"nameplate_capacity_mw"})
check(battery_small["plant_id"].equals(battery["Plant Code"]),"battery: plant identifiers preserved")
same_distribution(battery,battery_small,["county_geoid"],"battery")
battery_ready = save_geo("Texas_Battery_Plants",battery_small,"battery",["Point"])
check(len(battery_ready)==len(battery),"battery: no geometry exclusions")
if EXPORT_POINT_CSV: save_csv("Texas_Battery_Plants_Points",battery_ready,True)
print("Battery plants:",len(battery_ready))
del battery,battery_small,battery_ready

Optional columns not available: ['Plant Name', 'County']
Saved Texas_Battery_Plants: 200 mapped records
Saved Texas_Battery_Plants_Points: 200 rows
Battery plants: 200


## 4. Solar polygons and unique mapped footprint

In [6]:
solar = gpd.read_file(SOURCES["solar_parts"])
required(solar,["solar_record_id","county_geoid","county_solar_acres"],"solar")
check(len(solar)==EXPECTED["solar_project_parts"],"solar: county pieces",len(solar))
check(solar["solar_record_id"].nunique()==EXPECTED["solar_projects"],
      "solar: original projects represented",solar["solar_record_id"].nunique())
solar["county_solar_acres"] = pd.to_numeric(solar["county_solar_acres"],errors="raise")
check(solar["county_solar_acres"].notna().all() and (solar["county_solar_acres"]>0).all(),
      "solar: positive area")
check(np.isclose(solar["county_solar_acres"].sum(),EXPECTED_SOLAR_PROJECT_ACRES,atol=0.1),
      "solar: project acres reconcile",solar["county_solar_acres"].sum())
fields = ["solar_record_id","case_id","p_name","p_year","p_cap_ac","p_cap_dc",
          "polygon_acres","county_geoid","county_name","county_solar_acres"]
solar_small = pick(solar,fields,{"p_name":"project_name"})
same_distribution(solar,solar_small,["solar_record_id","county_geoid"],"solar")
solar_ready = save_geo("Texas_Solar_County_Project_Polygons",solar_small,"solar_parts",["Polygon","MultiPolygon"])
check(len(solar_ready)==len(solar),"solar: no geometry exclusions")
save_csv("Texas_Solar_County_Project_Attributes",solar_ready)
unique = gpd.read_file(SOURCES["solar_unique"])
required(unique,["county_geoid","unique_solar_acres"],"solar unique")
identifier_audit(unique,"county_geoid","solar unique")
unique["unique_solar_acres"] = pd.to_numeric(unique["unique_solar_acres"],errors="raise")
check((unique["unique_solar_acres"]>=0).all(),"solar: nonnegative unique footprint")
check(unique["unique_solar_acres"].sum()<=solar["county_solar_acres"].sum()+0.01,
      "solar: unique footprint no larger than project sum")
unique_ready = save_geo("Texas_Solar_County_Unique_Footprint",unique,"solar_unique",["Polygon","MultiPolygon"])
check(len(unique_ready)==len(unique),"solar: unique polygons no exclusions")
if SOURCES["solar_summary"].exists():
    summary = pd.read_csv(SOURCES["solar_summary"],dtype={"county_geoid":"string"})
    check(np.isclose(summary["summed_project_acres"].sum(),solar["county_solar_acres"].sum(),atol=0.01),
          "solar: Notebook 02 county summary acreage")
    save_csv("Texas_Solar_County_Summary",summary)
print("Solar project pieces:",len(solar),"distinct project records:",solar["solar_record_id"].nunique())
del solar,solar_small,solar_ready,unique,unique_ready

Saved Texas_Solar_County_Project_Polygons: 197 mapped records
Saved Texas_Solar_County_Project_Attributes: 197 rows
Saved Texas_Solar_County_Unique_Footprint: 86 mapped records
Saved Texas_Solar_County_Summary: 86 rows
Solar project pieces: 197 distinct project records: 187


## 5. RRC well surface location

In [7]:
#data checker you will get an error, ignore it and continue running the code
wells = gpd.read_file(SOURCES["wells"])
required(wells,["source_row_id","SYMNUM","well_category","county_geoid","county_match_status"],"wells")
identifier_audit(wells,"source_row_id","wells")
check(len(wells)==EXPECTED["wells"],"wells: complete Notebook 02 inventory",len(wells))
wells["symnum_normalized"] = pd.to_numeric(wells["SYMNUM"],errors="coerce").astype("Int64")
check(int(wells["symnum_normalized"].eq(3).fillna(False).sum())==288905,
      "wells: dry holes retained",int(wells["symnum_normalized"].eq(3).fillna(False).sum()))
fields = ["source_row_id","SURFACE_ID","API","WELLID","SYMNUM","symnum_normalized",
          "well_category","county_geoid","county_name","county_match_status",
          "within_county_boundary","legacy_excluded_category","well_scenario_label"]
wells_small = pick(wells,fields)
check(wells_small["source_row_id"].equals(wells["source_row_id"]),"wells: source identifiers preserved")
same_distribution(wells,wells_small,["SYMNUM","county_geoid"],"wells")
well_ready = save_geo("Texas_Wells_All_Categories",wells_small,"wells",["Point"])
check(len(well_ready)==len(wells),"wells: no unexpected geometry exclusions")
check(int(well_ready["county_geoid"].isna().sum())==1555,
      "wells: Notebook 02 unmatched count",int(well_ready["county_geoid"].isna().sum()))

Saved Texas_Wells_All_Categories: 1,380,352 mapped records


In [8]:
# I verify that the optimized master well dataset preserves all records
# and classifications from the original Notebook 02 inventory.

import geopandas as gpd

master_path = GEO / "Texas_Wells_All_Categories.parquet"

# I confirm that the master GeoParquet file was successfully exported.
assert master_path.exists(), f"Missing master well dataset: {master_path}"

# I reload the saved file to verify the actual output on disk.
master_geo = gpd.read_parquet(master_path)

# I verify that the saved dataset contains all expected well records.
assert len(master_geo) == EXPECTED["wells"], (
    f"Expected {EXPECTED['wells']:,} wells, "
    f"but found {len(master_geo):,}"
)

# I check that every source record has a unique identifier.
assert master_geo["source_row_id"].notna().all()
assert master_geo["source_row_id"].is_unique

# I verify that the original SYMNUM classifications are preserved.
assert "SYMNUM" in master_geo.columns
assert "symnum_normalized" in master_geo.columns
assert "well_category" in master_geo.columns

# I compare the saved records with the dataset prepared in Cell 12.
assert set(master_geo["source_row_id"]) == set(
    well_ready["source_row_id"]
)

# I check that the classification distribution has not changed.
original_counts = (
    well_ready["symnum_normalized"]
    .value_counts(dropna=False)
    .sort_index()
)

saved_counts = (
    master_geo["symnum_normalized"]
    .value_counts(dropna=False)
    .sort_index()
)

pd.testing.assert_series_equal(
    original_counts,
    saved_counts,
    check_names=False
)

# I summarize the classifications retained in the master dataset.
print("Master well dataset validation")
print(f"Total wells: {len(master_geo):,}")
print(f"Unique identifiers: {master_geo['source_row_id'].nunique():,}")
print(f"SYMNUM classifications: {master_geo['symnum_normalized'].nunique(dropna=True)}")
print(f"Missing SYMNUM values: {master_geo['symnum_normalized'].isna().sum():,}")

print("\nPASS: Master well dataset is complete and classifications are preserved.")

Master well dataset validation
Total wells: 1,380,352
Unique identifiers: 1,380,352
SYMNUM classifications: 54
Missing SYMNUM values: 0

PASS: Master well dataset is complete and classifications are preserved.


## 6. Pipelines: preserve source codes and county-clipped mileage

In [9]:
pipeline = gpd.read_file(SOURCES["pipeline_parts"])
required(pipeline,["pipeline_record_id","county_geoid","county_pipeline_miles","STATUS_CD","SYSTYPE"],"pipelines")
identifier_audit(pipeline,"pipeline_record_id","pipelines",unique=False)
check(pipeline["pipeline_record_id"].nunique()<=EXPECTED["pipeline_source_records"],
      "pipelines: clipped source IDs do not exceed original source count")
pipeline["county_pipeline_miles"] = pd.to_numeric(pipeline["county_pipeline_miles"],errors="raise")
check(pipeline["county_pipeline_miles"].notna().all() and (pipeline["county_pipeline_miles"]>0).all(),
      "pipelines: positive mapped lengths")
check(np.isclose(pipeline["county_pipeline_miles"].sum(),EXPECTED_PIPELINE_COUNTY_MILES,atol=0.1),
      "pipelines: Notebook 02 county-allocated mileage",pipeline["county_pipeline_miles"].sum())
pipeline["status_code"] = pipeline["STATUS_CD"].astype("string").str.strip().str.upper()
pipeline["system_code"] = pipeline["SYSTYPE"].astype("string").str.strip().str.upper()
fields = ["pipeline_record_id","TPMS_ID","STATUS_CD","status_code","status_description",
          "SYSTYPE","system_code","system_description","OPER_NM","SYS_NM","DIAMETER",
          "CMDTY_DESC","ALBERS_MIL","county_geoid","county_name","county_pipeline_miles"]
pipeline_small = pick(pipeline,fields)
same_distribution(pipeline,pipeline_small,["pipeline_record_id","county_geoid","STATUS_CD","SYSTYPE"],"pipeline")
check(np.isclose(pipeline_small["county_pipeline_miles"].sum(),pipeline["county_pipeline_miles"].sum(),atol=1e-6),
      "pipeline: preserved clipped mileage")
pipeline_ready = save_geo("Texas_Pipeline_County_Pieces_All_Statuses",pipeline_small,"pipeline",["LineString","MultiLineString"])
check(len(pipeline_ready)==len(pipeline),"pipeline: no unexpected geometry exclusions")

Saved Texas_Pipeline_County_Pieces_All_Statuses: 507,807 mapped records


In [10]:
result = pd.DparataFrame(EXPORT_LOG)
check((result["mapped_rows"]+result["non_map_ready_rows"]==result["source_rows"]).all(),
      "final: every export row accounted for")
check(not result["category"].isin(["fossil","refinery","uic"]).any(),
      "final: excluded infrastructure absent")
result.to_csv(AUDIT / "Notebook03_Export_Reconciliation.csv",index=False)
checks = pd.DparataFrame(CHECKS)
checks.to_csv(AUDIT / "Notebook03_Validation_Checks.csv",index=False)
manifest = {
    "notebook": "03_Texas_Energy_Cleaning_Optimization_Corrected",
    "scope": ["wind turbines","solar facilities","battery energy storage","RRC surface wells","pipelines"],
    "excluded": ["fossil power plants","refineries","separate UIC disposal well dataset"],
    "input_files": [{"name":k,"path":str(p),"exists":p.exists(),
                     "size_bytes":p.stat().st_size if p.is_file() else None} for k,p in SOURCES.items()],
    "exports": EXPORT_LOG,
    "validation_passed": bool(checks["passed"].all()),
    "acreage_estimated": False,
}
(AUDIT / "Notebook03_Manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
display(result)
print("I completed Notebook 03 validation:",len(checks),"checks passed.")

AttributeError: module 'pandas' has no attribute 'DparataFrame'